In [ ]:
import pandas as pd

# Load the datasets
matches_df = pd.read_csv('matches.csv')
deliveries_df = pd.read_csv('deliveries.csv')

# Check the column names to make sure we're using the right ones
print("Matches columns:", matches_df.columns.tolist())
print("Deliveries columns:", deliveries_df.columns.tolist())

# Assuming 'id' in matches.csv corresponds to 'match_id' in deliveries.csv
# Merge the datasets
merged_df = pd.merge(
    deliveries_df,
    matches_df,
    left_on='match_id',
    right_on='id',
    how='left'
)

# Check the shape of the merged dataframe
print("Original deliveries shape:", deliveries_df.shape)
print("Original matches shape:", matches_df.shape)
print("Merged dataframe shape:", merged_df.shape)

# Save the merged dataset
merged_df.to_csv('merged_ipl_data.csv', index=False)

Matches columns: ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']
Deliveries columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']
Original deliveries shape: (260920, 17)
Original matches shape: (1095, 20)
Merged dataframe shape: (260920, 37)


In [ ]:
import pandas as pd

# Load the dataset
file_path = "merged_ipl_data.csv"  # Update with your actual file path

# Define required columns (Keep only these)
required_columns = [
    "match_id", "id", "date", "team1", "team2", "toss_winner", "toss_decision",
    "winner", "result", "result_margin",
    "inning", "batting_team", "bowling_team", "over", "ball", "total_runs", "is_wicket"
]

try:
    # Read CSV with encoding detection
    df = pd.read_csv(file_path, encoding="ISO-8859-1")

    # Keep only required columns (drop all others)
    df_filtered = df[required_columns]

    # Save the reduced file
    reduced_file_path = "reduced_ipl_data.csv"
    df_filtered.to_csv(reduced_file_path, index=False)

    print(f"Reduced dataset saved as {reduced_file_path}")
    print(df_filtered.head())  # Show first few rows
except Exception as e:
    print("Error:", e)


Reduced dataset saved as reduced_ipl_data.csv
   match_id      id        date                        team1  \
0    335982  335982  2008-04-18  Royal Challengers Bangalore   
1    335982  335982  2008-04-18  Royal Challengers Bangalore   
2    335982  335982  2008-04-18  Royal Challengers Bangalore   
3    335982  335982  2008-04-18  Royal Challengers Bangalore   
4    335982  335982  2008-04-18  Royal Challengers Bangalore   

                   team2                  toss_winner toss_decision  \
0  Kolkata Knight Riders  Royal Challengers Bangalore         field   
1  Kolkata Knight Riders  Royal Challengers Bangalore         field   
2  Kolkata Knight Riders  Royal Challengers Bangalore         field   
3  Kolkata Knight Riders  Royal Challengers Bangalore         field   
4  Kolkata Knight Riders  Royal Challengers Bangalore         field   

                  winner result  result_margin  inning           batting_team  \
0  Kolkata Knight Riders   runs          140.0       1  Kolka

In [ ]:
import pandas as pd

# Load the dataset
file_path = "reduced_ipl_data.csv"
df = pd.read_csv(file_path)

# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

# Handling missing values
df["winner"].fillna("No Result", inplace=True)
df["result_margin"].fillna("N/A", inplace=True)

# Ensure data is sorted properly
df.sort_values(by=["match_id", "inning", "over", "ball"], inplace=True)

# Compute cumulative runs for each match and innings
df["cumulative_runs"] = df.groupby(["match_id", "inning"])["total_runs"].cumsum()

# Compute cumulative wickets for each match and innings
df["cumulative_wickets"] = df.groupby(["match_id", "inning"])["is_wicket"].cumsum()

# Compute overs completed as Over + (Ball - 1) / 6
df["overs_completed"] = df["over"] + (df["ball"] - 1) / 6

# Compute Current Run Rate (CRR)
df["current_run_rate"] = df["cumulative_runs"] / df["overs_completed"]

# Handle division by zero cases (first ball of the match)
df["current_run_rate"].fillna(0, inplace=True)

# Save the processed dataset
processed_file_path = "processed_ipl_data.csv"
df.to_csv(processed_file_path, index=False)

print(f"Processed dataset saved as {processed_file_path}")
print(df.head())  # Display first few rows

FileNotFoundError: [Errno 2] No such file or directory: 'reduced_ipl_data.csv'

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = "reduced_ipl_data.csv"  # Update with actual file path
df = pd.read_csv(file_path)

# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

# Handling missing values
df["winner"].fillna("No Result", inplace=True)
df["result_margin"].fillna("N/A", inplace=True)

# Ensure data is sorted properly for cumulative calculations
df.sort_values(by=["match_id", "inning", "over", "ball"], inplace=True)

# Compute cumulative runs for each match and innings
df["cumulative_runs"] = df.groupby(["match_id", "inning"])["total_runs"].cumsum()

# Compute cumulative wickets for each match and innings
df["cumulative_wickets"] = df.groupby(["match_id", "inning"])["is_wicket"].cumsum()

# Compute overs completed correctly (starting from 1 and using decimal format)
df["overs_completed"] = (df["over"] + 1) + ((df["ball"] - 1) / 10)

# Compute Current Run Rate (CRR)
df["current_run_rate"] = df["cumulative_runs"] / df["overs_completed"]

# Fix division by zero issues (replace inf and NaN values with 0)
df["current_run_rate"].replace([np.inf, -np.inf], 0, inplace=True)
df["current_run_rate"].fillna(0, inplace=True)

# Save the processed dataset
processed_file_path = "processed_ipl_data.csv"
df.to_csv(processed_file_path, index=False)

print(f"Processed dataset saved as {processed_file_path}")
print(df[["match_id", "inning", "over", "ball", "cumulative_runs", "cumulative_wickets", "overs_completed", "current_run_rate"]].head(20))  # Show sample data

Missing Values:
 match_id            0
id                  0
date                0
team1               0
team2               0
toss_winner         0
toss_decision       0
winner            490
result              0
result_margin    4124
inning              0
batting_team        0
bowling_team        0
over                0
ball                0
total_runs          0
is_wicket           0
dtype: int64


<ipython-input-3-14da8ba03708>:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["winner"].fillna("No Result", inplace=True)
<ipython-input-3-14da8ba03708>:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

Processed dataset saved as processed_ipl_data.csv
    match_id  inning  over  ball  cumulative_runs  cumulative_wickets  \
0     335982       1     0     1                1                   0   
1     335982       1     0     2                1                   0   
2     335982       1     0     3                2                   0   
3     335982       1     0     4                2                   0   
4     335982       1     0     5                2                   0   
5     335982       1     0     6                2                   0   
6     335982       1     0     7                3                   0   
7     335982       1     1     1                3                   0   
8     335982       1     1     2                7                   0   
9     335982       1     1     3               11                   0   
10    335982       1     1     4               17                   0   
11    335982       1     1     5               21                   0   
1

In [ ]:
import pandas as pd

# Load the IPL dataset
ipl_file_path = "processed_ipl_data_updated.csv"  # Update with the correct path if needed
ipl_df = pd.read_csv(ipl_file_path)

# Load the matches dataset (contains venue information)
matches_file_path = "matches-1.csv"  # Update with the correct path if needed
matches_df = pd.read_csv(matches_file_path)

# Check if the venue column exists
if "venue" in matches_df.columns:
    # Merge the venue column into the IPL dataset using match_id
    ipl_df = ipl_df.merge(matches_df[["id", "venue"]], on="id", how="left")

    # Save the updated dataset
    output_file_path = "processed_ipl_data_with_venue.csv"
    ipl_df.to_csv(output_file_path, index=False)

    print(f"Updated dataset with venue saved as: {output_file_path}")
else:
    print("Error: 'venue' column not found in matches dataset.")

Updated dataset with venue saved as: processed_ipl_data_with_venue.csv


In [ ]:
import pandas as pd

# Load the dataset
file_path = "Process.csv"  # Change this to your actual file path
df = pd.read_csv(file_path)

# Convert 'over' and 'ball' to numeric
df["over"] = pd.to_numeric(df["over"], errors="coerce")
df["ball"] = pd.to_numeric(df["ball"], errors="coerce")

# Initialize columns with "N/A"
df["remaining_runs"] = "N/A"
df["remaining_overs"] = "N/A"
df["required_run_rate"] = "N/A"

# Process each match separately
match_ids = df["match_id"].unique()

for match in match_ids:
    match_data = df[df["match_id"] == match]

    # Identify first batting team and their total runs
    first_batting_team = match_data["batting_team"].iloc[0]
    first_batting_runs = match_data[match_data["batting_team"] == first_batting_team]["total_runs"].sum()

    # Target for second batting team
    target = first_batting_runs + 1

    # Get second batting team mask
    second_batting_team_mask = (df["match_id"] == match) & (df["batting_team"] != first_batting_team)

    # Calculate remaining runs and overs
    df.loc[second_batting_team_mask, "remaining_runs"] = target - df.loc[second_batting_team_mask, "cumulative_runs"]
    df.loc[second_batting_team_mask, "remaining_overs"] = 20 - df.loc[second_batting_team_mask, "overs_completed"]

    # Calculate Required Run Rate (RRR) safely, avoiding division by zero
    # If remaining_overs is 0, set required_run_rate to NaN
    df.loc[second_batting_team_mask, "required_run_rate"] = df.loc[second_batting_team_mask].apply(
        lambda row: row["remaining_runs"] / row["remaining_overs"] if row["remaining_overs"] != 0 else float('nan'),
        axis=1
    )
    # Replace infinite values with NaN (redundant after the above change, but kept for safety)
    df.loc[df["required_run_rate"] == float("inf"), "required_run_rate"] = None

# Save the updated dataset
output_file_path = "processed_ipl_data_updated.csv"  # Change this to your desired output path
df.to_csv(output_file_path, index=False)

print(f"Updated dataset saved as: {output_file_path}")

Updated dataset saved as: processed_ipl_data_updated.csv
